![Logo del Proyecto](https://www.utdt.edu/Images/prensa/utdt-color-baja.png)

<h1 align="center"> Tesis - Master in Management + Analytics </h1>
<h2 align="center"> IA como Herramienta para el Control de Calidad de Imágenes<br> en el Diagnóstico de Retinopatía Diabética </h2>

**Fecha:** Octubre 2024<br>
**Autora:** Gabriela Moran<br>
**Tutor:** [Santiago Cisco](https://ar.linkedin.com/in/mariocisco)<br>

In [16]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import pickle
import configparser
from sklearn.model_selection import train_test_split

In [17]:
sys.path.append('./tools')

In [18]:
MiM_path = Path(os.getcwd()).parent
DIR = sys.path.append(MiM_path/'data')
BASE = Path(MiM_path/'data')
random_state = 221218

# Split en train, validation y test

## HRF

In [19]:
with open(BASE/'labels/HRF.pkl','rb') as f:
    HRF = pickle.load(f)

In [20]:
HRF_x = HRF.filename
HRF_y = HRF.label

In [21]:
HRF_x_train, HRF_x_rest, HRF_y_train, HRF_y_rest = train_test_split(HRF_x, HRF_y, test_size = 0.5, stratify = HRF_y, random_state=random_state)

In [22]:
HRF_x_val, HRF_x_test, HRF_y_val, HRF_y_test = train_test_split(HRF_x_rest,HRF_y_rest,test_size = 0.3,stratify = HRF_y_rest, random_state=random_state)

In [23]:
HRF_config = configparser.ConfigParser()
HRF_config['split'] = {'type':'holdout',
                    'training':','.join(HRF_x_train.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist()),
                    'validation':','.join(HRF_x_val.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist()),
                    'test':','.join(HRF_x_test.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/HRF.ini', 'w') as configfile:
    HRF_config.write(configfile)

## DDR

In [24]:
raw_DDR = Path(BASE/'raw-data/DDR-dataset/DR_grading')

In [25]:
DDR_train = pd.read_csv(raw_DDR/'train.txt',sep=' ', header = None)
DDR_val = pd.read_csv(raw_DDR/'valid.txt',sep=' ', header = None)
DDR_test = pd.read_csv(raw_DDR/'test.txt',sep=' ', header = None)

In [26]:
DDR_config = configparser.ConfigParser()
DDR_config['split'] = {'type':'holdout',
                    'training':','.join(DDR_train[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist()),
                    'validation':','.join(DDR_val[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist()),
                    'test':','.join(DDR_test[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/DDR.ini', 'w') as configfile:
    DDR_config.write(configfile)

## Kaggle

In [27]:
raw_kaggle = Path(BASE/'raw-data/Kaggle')

### Zhou

In [28]:
Zhou_train = pd.read_csv(raw_kaggle/'quality_label_train.csv',sep=',', header = 0)
Zhou_val = pd.read_csv(raw_kaggle/'quality_label_validate.csv',sep=',', header = 0)
Zhou_test = pd.read_csv(raw_kaggle/'quality_label_test.csv',sep=',', header = 0)

In [29]:
Zhou_train.shape

(35126, 2)

In [30]:
with open(BASE/'labels/Kaggle_zhou.pkl','rb') as f:
    Zhou = pickle.load(f)

In [31]:
# Elimina la imagen que esta dañana y no va a ser parte del entrenamiento
Zhou.filename = Zhou.filename.str.replace('.png','', regex=False)
Zhou_train = Zhou_train[Zhou_train.image.isin(Zhou.filename)]

In [32]:
Zhou_train.shape

(35125, 2)

In [33]:
Zhou_config = configparser.ConfigParser()
Zhou_config['split'] = {'type':'holdout',
                        'training':','.join(Zhou_train['image'].apply(lambda x: 'Kaggle/'+x).tolist()),
                        'validation':','.join(Zhou_val['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist()),
                        'test':','.join(Zhou_test['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/Zhou.ini', 'w') as configfile:
    Zhou_config.write(configfile)

## Deep Diabetic Retinopathy Image Dataset DeepDRiD

In [34]:
raw_DRiD = Path(BASE/'raw-data/Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-')

In [35]:
x = pd.read_csv(raw_DRiD/'regular_fundus_images/regular-fundus-training/regular-fundus-training.csv')
DRiD_train = x.loc[:,['image_id','Overall quality']]

In [36]:
x = pd.read_csv(raw_DRiD/'regular_fundus_images/regular-fundus-validation/regular-fundus-validation.csv')
DRiD_val = x.loc[:,['image_id','Overall quality']]

In [37]:
DRiD = pd.concat([DRiD_train,DRiD_val],ignore_index=True)

In [38]:
drid_config = configparser.ConfigParser()
drid_config['split'] = {'type':'holdout',
                    'training':'',
                    'validation':'',
                    'test':','.join(DRiD['image_id'].apply(lambda x: 'Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-/'+x).tolist())}

with open(BASE/'splits/DRID.ini', 'w') as configfile:
    drid_config.write(configfile)

## Global

In [39]:
# Target: Buena/Mala Calidad
global2_config = configparser.ConfigParser()
global2_config['split'] = {'type':'holdout',
                        'training':HRF_config['split']['training']+','+DDR_config['split']['training']+','+Zhou_config['split']['training'],
                        'validation':HRF_config['split']['validation']+','+DDR_config['split']['validation']+','+Zhou_config['split']['validation'],
                        'test':HRF_config['split']['test']+','+DDR_config['split']['test']+','+Zhou_config['split']['test']+','+drid_config['split']['test']
                        }

with open(BASE/'splits/global_binaria.ini', 'w') as configfile:
    global2_config.write(configfile)

# Labels
Creo un archivo de configuración global con el nombre del archivo y el label.

In [40]:
with open(BASE/'labels/HRF.pkl','rb') as f:
    HRF = pickle.load(f)

with open(BASE/'labels/DDR.pkl','rb') as f:
    DDR = pickle.load(f)

with open(BASE/'labels/Kaggle_zhou.pkl','rb') as f:
    zhou = pickle.load(f)

with open(BASE/'labels/DRiD.pkl','rb') as f:
    drid = pickle.load(f)

In [41]:
HRF['filename'] = HRF['filename'].str.replace('\..*','', regex=True)
DDR['filename'] = DDR['filename'].str.replace('\..*','', regex=True)
zhou['filename'] = zhou['filename'].str.replace('\..*','', regex=True)
drid['filename'] = drid['filename'].str.replace('\..*','', regex=True)

In [42]:
HRF.filename = HRF.filename.apply(lambda x: 'HRF - Quality/'+x)
DDR.filename = DDR.filename.apply(lambda x: 'DDR-dataset/'+x)
zhou.filename = zhou.filename.apply(lambda x: 'Kaggle/'+x)
drid.filename = drid.filename.apply(lambda x: 'Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-/'+x)

In [43]:
all_labels = pd.concat([HRF,DDR,zhou,drid], axis=0)

In [44]:
global_labels_config = configparser.ConfigParser()
global_labels_config.optionxform = str
global_labels_config['label'] = {row["filename"]: str(row["label"]) for _, row in all_labels.iterrows()}

with open(BASE/'labels/global_labels.ini', 'w') as configfile:
    global_labels_config.write(configfile)